# ArXiv RAG — Stage 2: Embedding Pipeline (Contextual Retrieval)

Reads the papers already in Neon Postgres (from Stage 1), chunks each abstract,
generates an LLM situating blurb per chunk (Anthropic's Contextual Retrieval),
embeds the blurb+chunk with `nomic-embed-text-v2-moe` on GPU, and bulk-inserts
into the `chunks` table (`content`, `context`, `embedding`).

**Works on both Kaggle and Google Colab** — the secrets-loading cell below detects which platform you're on.

## Before running — Colab setup

1. **Runtime → Change runtime type → T4 GPU** (top menu)
2. Click the **key icon 🔑** in the left sidebar (Secrets panel)
3. Add these secrets (toggle "Notebook access" on for each):
   - `DATABASE_URL` — `postgresql://neondb_owner:<password>@<host>/neondb?sslmode=require&channel_binding=require`
   - `GROQ_API_KEY` — your Groq key (used to generate context blurbs)
   - `GEMINI_API_KEY` — your Gemini key (fallback LLM)
4. **Runtime → Run all**

## Before running — Kaggle setup (unchanged)

1. **Add-ons → Secrets** — add `DATABASE_URL`, `GROQ_API_KEY`, `GEMINI_API_KEY`
2. **Settings → Accelerator → GPU T4**
3. **Run All**

## Important: this run adds context to ALL chunks, including previously-embedded ones

Since the `chunks` table migration (`db/migrations/001_contextual_retrieval.sql`) must already
be applied to Neon before running this notebook. This pipeline **deletes and re-inserts**
chunks for every paper processed (rather than skipping already-embedded papers) so that
existing chunks pick up their new `context` blurb. Expect ~2-3x longer runtime than the original
run because of the added LLM call per chunk — budget ~60-90 min for 10k papers on a T4.

In [ ]:
# Install dependencies
!pip install -q 'psycopg[binary]' psycopg_pool 'pgvector>=0.3.5' sentence-transformers einops numpy openai

In [ ]:
import os

def _load_secret(name: str) -> str | None:
    """Load a secret from Colab userdata, Kaggle secrets, or existing env var."""
    # Try Colab
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    # Try Kaggle
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

for _key in ("DATABASE_URL", "GROQ_API_KEY", "GEMINI_API_KEY"):
    _val = _load_secret(_key)
    if _val:
        os.environ[_key] = _val
        print(f"{_key} loaded")
    else:
        print(f"WARNING: {_key} not found — set it manually: os.environ['{_key}'] = '...'")

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import logging
import time
import re
from dataclasses import dataclass

from psycopg_pool import AsyncConnectionPool
from pgvector.psycopg import register_vector_async
from sentence_transformers import SentenceTransformer

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

# ── Config ──────────────────────────────────────────────────────────────────
EMBED_MODEL   = 'nomic-ai/nomic-embed-text-v2-moe'
EMBED_DIM     = 768
CHUNK_SIZE    = 400   # tokens (approx words)
CHUNK_OVERLAP = 50
BATCH_SIZE    = 1000  # papers per flush
EMBED_BATCH   = 512   # texts per model.encode() call
LIMIT         = 50_000

print('Config loaded')

In [ ]:
# ── Chunking ─────────────────────────────────────────────────────────────────
@dataclass
class Chunk:
    doc_id: str
    section_title: str
    chunk_index: int
    content: str
    token_count: int
    context: str = ""  # LLM-generated situating blurb

def _word_chunks(text: str, size: int, overlap: int) -> list[str]:
    words = text.split()
    if not words:
        return []
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + size, len(words))
        chunks.append(' '.join(words[start:end]))
        if end == len(words):
            break
        start += size - overlap
    return chunks

def chunk_text(text: str, doc_id: str, section_title: str = 'abstract') -> list[Chunk]:
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    raw = _word_chunks(text, CHUNK_SIZE, CHUNK_OVERLAP)
    return [
        Chunk(doc_id=doc_id, section_title=section_title,
              chunk_index=i, content=c, token_count=len(c.split()))
        for i, c in enumerate(raw)
    ]

print('Chunking helpers ready')

In [ ]:
# ── Embedding model ───────────────────────────────────────────────────────────
print(f'Loading {EMBED_MODEL} ...')
_model = SentenceTransformer(EMBED_MODEL, trust_remote_code=True, device=device)
print(f'Model loaded (dim={EMBED_DIM})')

def embed_chunks(chunks: list[Chunk]) -> list[tuple[Chunk, list[float]]]:
    if not chunks:
        return []
    texts = [
        f'search_document: {c.context}\n\n{c.content}' if c.context else f'search_document: {c.content}'
        for c in chunks
    ]
    embeddings = _model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=EMBED_BATCH,
        device=device,
    )
    return list(zip(chunks, embeddings.tolist()))

# Quick smoke test
test = embed_chunks([Chunk('test', 'abstract', 0, 'attention is all you need', 5)])
assert len(test[0][1]) == EMBED_DIM
print(f'Smoke test passed — embedding dim={len(test[0][1])}')

In [ ]:
# ── Context blurb generation (Anthropic Contextual Retrieval) ────────────────
import asyncio
import time
from openai import AsyncOpenAI, RateLimitError, APIStatusError

_groq_client = AsyncOpenAI(api_key=os.environ.get('GROQ_API_KEY', ''), base_url='https://api.groq.com/openai/v1')
_gemini_client = AsyncOpenAI(api_key=os.environ.get('GEMINI_API_KEY', ''), base_url='https://generativelanguage.googleapis.com/v1beta/openai/')

GROQ_MODEL = 'llama-3.3-70b-versatile'
GEMINI_MODEL = 'gemini-2.5-flash'
CONTEXT_MAX_TOKENS = 100

# Free-tier pacing: Gemini free tier allows 5 requests/minute (12s/req).
# Groq's 100k tokens/day free budget resets once every 24h — once we see a
# "tokens per day" 429, retrying immediately is pointless, so we stop
# calling Groq for the rest of this session.
GEMINI_MIN_INTERVAL_SECONDS = 13.0
_groq_daily_quota_exhausted = False
_last_gemini_call_at = 0.0

_CONTEXT_PROMPT = """\
<document>
{abstract}
</document>

Here is the chunk to situate:
<chunk>
{chunk}
</chunk>

Write 1-2 sentences that situate this chunk within the paper titled "{title}". \
Focus on what aspect of the paper this chunk covers. Be concise and do not repeat \
the chunk verbatim."""

async def generate_context(title: str, abstract: str, chunk: str) -> str:
    """Generate a situating blurb via Groq, falling back to paced Gemini calls."""
    global _groq_daily_quota_exhausted, _last_gemini_call_at

    prompt = _CONTEXT_PROMPT.format(abstract=abstract, chunk=chunk, title=title)
    messages = [{'role': 'user', 'content': prompt}]

    if not _groq_daily_quota_exhausted:
        try:
            resp = await _groq_client.chat.completions.create(
                model=GROQ_MODEL, messages=messages, temperature=0.0, max_tokens=CONTEXT_MAX_TOKENS,
            )
            return (resp.choices[0].message.content or '').strip()
        except (RateLimitError, APIStatusError) as e:
            msg = str(e)
            if 'tokens per day' in msg or 'TPD' in msg:
                logger.warning('Groq daily token quota exhausted — skipping Groq for rest of session')
                _groq_daily_quota_exhausted = True
            else:
                logger.warning('Groq error (%s), falling back to Gemini', e)
        except Exception as e:
            logger.warning('Groq error (%s), falling back to Gemini', e)

    # Pace Gemini calls to respect the free-tier 5 req/min limit
    elapsed = time.monotonic() - _last_gemini_call_at
    if elapsed < GEMINI_MIN_INTERVAL_SECONDS:
        await asyncio.sleep(GEMINI_MIN_INTERVAL_SECONDS - elapsed)

    try:
        resp = await _gemini_client.chat.completions.create(
            model=GEMINI_MODEL, messages=messages, temperature=0.0, max_tokens=CONTEXT_MAX_TOKENS,
        )
        return (resp.choices[0].message.content or '').strip()
    except Exception as e:
        logger.warning('Gemini also failed (%s) — embedding chunk without context', e)
        return ''
    finally:
        _last_gemini_call_at = time.monotonic()

async def generate_contexts_for_paper(title: str, abstract: str, chunks: list[Chunk]) -> None:
    """Populate chunk.context in-place for every chunk of one paper (sequential, rate-limit friendly)."""
    for c in chunks:
        c.context = await generate_context(title=title, abstract=abstract, chunk=c.content)

# Smoke test
_test_chunks = [Chunk('test', 'abstract', 0, 'the encoder maps an input sequence to a continuous representation', 10)]
await generate_contexts_for_paper('Attention Is All You Need', 'We propose a new architecture based on attention.', _test_chunks)
print('Context smoke test:', repr(_test_chunks[0].context))
print('Groq daily quota exhausted:', _groq_daily_quota_exhausted)

In [ ]:
# ── Database ──────────────────────────────────────────────────────────────────
_pool = None

async def init_pool():
    global _pool
    if _pool is not None:
        await _pool.close()
    url = os.environ['DATABASE_URL']
    _pool = AsyncConnectionPool(conninfo=url, min_size=1, max_size=5, open=False)
    await _pool.open()
    logger.info('DB pool initialized')

async def close_pool():
    global _pool
    if _pool:
        await _pool.close()

async def delete_chunks_for_papers(conn, paper_ids: list[int]):
    """Delete existing chunks for these papers so re-embedding with context doesn't duplicate rows."""
    if not paper_ids:
        return
    async with conn.cursor() as cur:
        await cur.execute('DELETE FROM chunks WHERE paper_id = ANY(%s)', (paper_ids,))

async def insert_chunks_batch(conn, chunks: list[dict]):
    await register_vector_async(conn)
    sql = """
        INSERT INTO chunks (paper_id, section_title, chunk_index, content, context, token_count, embedding)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """
    params = [
        (c['paper_id'], c['section_title'], c['chunk_index'],
         c['content'], c.get('context') or None, c['token_count'], c['embedding'])
        for c in chunks
    ]
    async with conn.cursor() as cur:
        await cur.executemany(sql, params)

async def write_batch_with_retry(paper_ids: list[int], rows: list[dict], max_attempts: int = 2):
    """Delete+insert+commit one batch. Neon's pooler can drop idle connections during
    long LLM rate-limit waits, so on failure we reinitialize the pool and retry once
    (a plain loop, not a re-entered context manager — asynccontextmanager only yields once)."""
    last_exc = None
    for attempt in range(max_attempts):
        try:
            async with _pool.connection() as conn:
                await delete_chunks_for_papers(conn, paper_ids)
                await insert_chunks_batch(conn, rows)
                await conn.commit()
            return
        except Exception as e:
            last_exc = e
            logger.warning('DB write failed (attempt %d/%d): %s', attempt + 1, max_attempts, e)
            if attempt < max_attempts - 1:
                await init_pool()
    raise last_exc

print('DB helpers ready')

In [ ]:
# ── Main pipeline ─────────────────────────────────────────────────────────────
async def run_pipeline(limit: int = LIMIT, batch_size: int = BATCH_SIZE, contextual: bool = True):
    await init_pool()

    total_docs = 0
    total_chunks = 0
    start = time.time()
    buffer: list[tuple[int, str, str, list]] = []  # (paper_id, title, abstract, chunks)

    async def flush(buf):
        if not buf:
            return 0

        # Generate context blurbs sequentially per paper (rate-limit friendly)
        if contextual:
            for pid, title, abstract, chunks in buf:
                await generate_contexts_for_paper(title, abstract, chunks)

        paper_ids_flat, chunks_flat = [], []
        for pid, _title, _abstract, chunks in buf:
            for c in chunks:
                paper_ids_flat.append(pid)
                chunks_flat.append(c)

        pairs = embed_chunks(chunks_flat)

        rows = [
            {
                'paper_id': paper_ids_flat[i],
                'section_title': chunk.section_title,
                'chunk_index': chunk.chunk_index,
                'content': chunk.content,
                'context': chunk.context,
                'token_count': chunk.token_count,
                'embedding': emb,
            }
            for i, (chunk, emb) in enumerate(pairs)
        ]

        unique_paper_ids = list({pid for pid, _, _, _ in buf})
        await write_batch_with_retry(unique_paper_ids, rows)

        return len(rows)

    async with _pool.connection() as conn:
        async with conn.cursor() as cur:
            await cur.execute(
                """
                SELECT id, arxiv_id, title, abstract
                FROM papers
                WHERE abstract IS NOT NULL AND abstract != ''
                ORDER BY published_at DESC
                LIMIT %s
                """,
                (limit,),
            )
            rows_fetched = await cur.fetchall()

    logger.info('Fetched %d papers from DB', len(rows_fetched))

    for paper_id, arxiv_id, title, abstract in rows_fetched:
        chunks = chunk_text(abstract, doc_id=arxiv_id)
        if not chunks:
            continue

        buffer.append((paper_id, title, abstract, chunks))
        total_docs += 1

        if len(buffer) >= batch_size:
            n = await flush(buffer)
            total_chunks += n
            buffer = []
            elapsed = time.time() - start
            rate = total_docs / elapsed * 60
            logger.info(
                'Processed %d docs | %d chunks | %.0f docs/min | %.1fs elapsed',
                total_docs, total_chunks, rate, elapsed,
            )

    # Flush remainder
    n = await flush(buffer)
    total_chunks += n

    await close_pool()

    elapsed = time.time() - start
    stats = {
        'total_docs': total_docs,
        'total_chunks': total_chunks,
        'elapsed_minutes': round(elapsed / 60, 1),
    }
    logger.info('Pipeline complete: %s', stats)
    return stats

print('Pipeline function defined — ready to run')

In [ ]:
# ── RUN ───────────────────────────────────────────────────────────────────────
# Reality check on free-tier throughput:
#   - Groq: 100k tokens/day free budget. Once exhausted (checked automatically),
#     every chunk falls through to Gemini for the rest of the day.
#   - Gemini free tier: 5 requests/minute -> paced to ~4.6/min here.
# At Gemini-only speed, a single Colab session can realistically process ~250-350
# papers/hour. Set `limit` to what you can afford to run today; re-run this cell
# on subsequent days (Groq quota resets every 24h) to cover more of the corpus.
stats = await run_pipeline(limit=300, batch_size=50, contextual=True)
print('\n✓ Done:', stats)

# To process the rest of the corpus without LLM blurbs (fast, no rate limits),
# run again with contextual=False - this embeds chunks with plain content only:
#   stats = await run_pipeline(limit=10_000, batch_size=1000, contextual=False)